# Gemini Semantic Profile 기반 JD 매칭 고도화 v1

이 노트북은 기존 `baseline_faiss_matching.ipynb`를 직접 수정하지 않고, Gemini 기반 구조화 전처리와 100쌍 벤치마크 평가를 추가한 독립 실험 노트북입니다.

핵심 흐름은 다음과 같습니다.

1. `user_data.csv`를 `userId` 단위로 병합하되, 같은 자기소개서가 경험 행마다 반복 가중되지 않도록 중복을 제거합니다.
2. `company_jobdescription.csv`에서 deterministic 1,000개 JD 샘플을 구성합니다.
3. Gemini structured output으로 User/JD를 동일한 Semantic Profile 스키마로 정리합니다.
4. API 키나 `google-genai` 패키지가 없으면 실험 구조 검증이 가능하도록 deterministic heuristic profile로 대체합니다.
5. 기존 원문 임베딩 baseline, Gemini profile 임베딩 모델, profile-weighted score 모델을 비교합니다.
6. 10명 유저 × 후보 JD 10개 = 100쌍 벤치마크를 만들고 NDCG@K, Precision@K, MRR@K, Spearman correlation을 계산합니다.

주의: 실제 Gemini 결과를 쓰려면 `google-genai` 설치와 `GEMINI_API_KEY` 또는 `GOOGLE_API_KEY` 환경변수 설정이 필요합니다.

In [ ]:
# 필요한 패키지 임포트
import os
import sys
import re
import json
import math
import time
import hashlib
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

try:
  import faiss
  HAS_FAISS = True
except Exception as exc:
  HAS_FAISS = False
  FAISS_IMPORT_ERROR = exc

try:
  from sentence_transformers import SentenceTransformer
  HAS_SENTENCE_TRANSFORMERS = True
except Exception as exc:
  HAS_SENTENCE_TRANSFORMERS = False
  SENTENCE_TRANSFORMERS_IMPORT_ERROR = exc

try:
  from google import genai
  from google.genai import types
  HAS_GENAI = True
except Exception as exc:
  HAS_GENAI = False
  GENAI_IMPORT_ERROR = exc

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 160)
pd.set_option('display.max_columns', 80)

print('faiss:', HAS_FAISS)
print('sentence_transformers:', HAS_SENTENCE_TRANSFORMERS)
print('google-genai:', HAS_GENAI)


## 0. 설정

실험 재현성을 위해 샘플링 seed와 출력 경로를 한 곳에서 관리합니다. `GEMINI_MODEL`은 기본값을 `gemini-3-flash-preview`로 두었습니다. 실제 사용 가능한 `gemini-3.1-flash` 계열 모델명이 있다면 환경변수나 아래 변수만 바꾸면 됩니다.

API 키는 코드/노트북/CSV에 저장하지 않습니다. `GEMINI_API_KEY` 또는 `GOOGLE_API_KEY` 환경변수를 권장하고, Jupyter에서 직접 실행할 때만 `getpass`로 입력받습니다.

In [ ]:
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
OUTPUT_DIR = DATA_DIR / 'gemini_profile_outputs'
CACHE_DIR = DATA_DIR / 'gemini_cache'

USER_DATA_PATH = DATA_DIR / 'user_data.csv'
JD_DATA_PATH = DATA_DIR / 'company_jobdescription.csv'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
JD_SAMPLE_SIZE = int(os.getenv('JD_SAMPLE_SIZE', '1000'))
BENCHMARK_USERS = int(os.getenv('BENCHMARK_USERS', '10'))
CANDIDATES_PER_USER = int(os.getenv('CANDIDATES_PER_USER', '10'))
GEMINI_MAX_WORKERS = int(os.getenv('GEMINI_MAX_WORKERS', '1'))

GEMINI_MODEL = os.getenv('GEMINI_MODEL', 'gemini-3-flash-preview')
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY') or os.getenv('GOOGLE_API_KEY')

IN_NOTEBOOK = 'ipykernel' in sys.modules
PROMPT_FOR_GEMINI_KEY = os.getenv('PROMPT_FOR_GEMINI_KEY', '1') == '1'
if not GEMINI_API_KEY and HAS_GENAI and IN_NOTEBOOK and PROMPT_FOR_GEMINI_KEY:
  try:
    from getpass import getpass
    entered_key = getpass('GEMINI_API_KEY를 입력하세요. 입력값은 화면에 표시되지 않습니다: ')
    GEMINI_API_KEY = entered_key.strip()
  except Exception as exc:
    print(f'키 입력을 사용할 수 없는 실행 환경입니다. 환경변수를 설정하지 않으면 fallback으로 실행됩니다: {type(exc).__name__}')

USE_GEMINI = bool(GEMINI_API_KEY) and HAS_GENAI
PROFILE_ENGINE = GEMINI_MODEL if USE_GEMINI else 'heuristic_fallback'

EMBEDDING_MODEL_NAME = os.getenv('EMBEDDING_MODEL_NAME', 'snunlp/KR-SBERT-V40K-klueNLI-augSTS')

print('PROJECT_ROOT:', PROJECT_ROOT)
print('GEMINI_MODEL:', GEMINI_MODEL)
print('USE_GEMINI:', USE_GEMINI)
print('PROFILE_ENGINE:', PROFILE_ENGINE)
print('JD_SAMPLE_SIZE:', JD_SAMPLE_SIZE, 'BENCHMARK_USERS:', BENCHMARK_USERS, 'CANDIDATES_PER_USER:', CANDIDATES_PER_USER)
print('GEMINI_MAX_WORKERS:', GEMINI_MAX_WORKERS)
if not HAS_GENAI:
  print('google-genai가 설치되어 있지 않습니다. 실제 Gemini 호출 전 다음 명령을 실행하세요: pip install google-genai')
if not GEMINI_API_KEY:
  print('GEMINI_API_KEY/GOOGLE_API_KEY가 없어 heuristic profile fallback으로 실행됩니다.')


## 1. 공통 유틸리티

아래 함수들은 텍스트 정리, 캐시 키 생성, 안전한 JSON 저장/로드, 간단한 키워드 추출을 담당합니다. Gemini 호출은 비용이 발생하므로 같은 입력은 캐시에 저장해 재사용합니다.

In [ ]:
def normalize_text(value: Any, max_chars: Optional[int] = None) -> str:
  if value is None or (isinstance(value, float) and np.isnan(value)):
    return ''
  text = str(value)
  text = re.sub(r'\s+', ' ', text).strip()
  if max_chars is not None and len(text) > max_chars:
    return text[:max_chars].rstrip()
  return text


def stable_hash(*parts: Any) -> str:
  raw = '\n---\n'.join(normalize_text(part) for part in parts)
  return hashlib.sha256(raw.encode('utf-8')).hexdigest()[:24]


def unique_keep_order(values: Iterable[Any], max_items: Optional[int] = None) -> List[str]:
  seen = set()
  result = []
  for value in values:
    text = normalize_text(value)
    if not text or text in seen:
      continue
    seen.add(text)
    result.append(text)
    if max_items is not None and len(result) >= max_items:
      break
  return result


def unique_join(values: Iterable[Any], sep: str = ' | ', max_items: Optional[int] = None) -> str:
  return sep.join(unique_keep_order(values, max_items=max_items))


def load_json(path: Path) -> Optional[Dict[str, Any]]:
  if not path.exists():
    return None
  with path.open('r', encoding='utf-8') as f:
    return json.load(f)


def save_json(path: Path, payload: Dict[str, Any]) -> None:
  path.parent.mkdir(parents=True, exist_ok=True)
  with path.open('w', encoding='utf-8') as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)


def safe_list(value: Any) -> List[str]:
  if value is None:
    return []
  if isinstance(value, list):
    return [normalize_text(v) for v in value if normalize_text(v)]
  if isinstance(value, str):
    if not value.strip():
      return []
    return [normalize_text(v) for v in re.split(r'[,|/;·•]', value) if normalize_text(v)]
  return [normalize_text(value)]


def jaccard(left: Iterable[Any], right: Iterable[Any]) -> float:
  left_set = {normalize_text(v).lower() for v in left if normalize_text(v)}
  right_set = {normalize_text(v).lower() for v in right if normalize_text(v)}
  if not left_set or not right_set:
    return 0.0
  return len(left_set & right_set) / len(left_set | right_set)


def bounded(value: float, low: float = 0.0, high: float = 1.0) -> float:
  if pd.isna(value):
    return low
  return float(max(low, min(high, value)))


In [ ]:
JOB_TAXONOMY = [
  'planning_strategy', 'management_support', 'software_dev', 'sales', 'marketing', 'data_ai_ml',
  'rnd', 'manufacturing', 'engineering_hw', 'service_pm', 'quality', 'finance_investment',
  'design', 'cs_cx', 'consulting', 'logistics_scm', 'procurement', 'education', 'unknown'
]

INDUSTRY_TAXONOMY = [
  'it_software', 'finance_fintech', 'ai_data', 'retail_ecommerce', 'bio_healthcare', 'semiconductor',
  'automotive', 'beauty_cosmetics', 'electronics', 'aerospace_defense', 'logistics', 'battery_energy',
  'machinery_heavy', 'chemical_materials', 'construction', 'fashion', 'medical', 'robotics', 'unknown'
]

JOB_KEYWORDS = {
  'software_dev': ['백엔드', '프론트엔드', '개발', '서버', '앱', '웹', 'react', 'spring', 'java', 'python'],
  'data_ai_ml': ['데이터', '머신러닝', 'ai', 'ml', '분석가', 'data scientist', '딥러닝'],
  'marketing': ['마케팅', '브랜드', '퍼포먼스', '콘텐츠', '광고', 'pr'],
  'sales': ['영업', '세일즈', 'business development', 'bd', '거래처'],
  'planning_strategy': ['기획', '전략', '사업기획', '서비스기획', 'pm', 'po'],
  'management_support': ['인사', 'hr', '총무', '경영지원', '사무', '운영지원'],
  'finance_investment': ['재무', '회계', '자금', '투자', '세무', '정산'],
  'quality': ['품질', 'qa', 'qc', '검사', '테스트', 'validation'],
  'manufacturing': ['생산', '제조', '공정', '설비', '라인', '생산관리'],
  'engineering_hw': ['하드웨어', '회로', '전장', '기구', 'cad', 'pcb', '펌웨어'],
  'rnd': ['연구', '개발연구', 'r&d', '연구개발', 'scientist'],
  'logistics_scm': ['물류', 'scm', '구매물류', '재고', '운송', '수출입'],
  'procurement': ['구매', '조달', '소싱', 'procurement'],
  'design': ['디자인', 'ui', 'ux', '그래픽', '영상', '콘텐츠 디자인'],
  'cs_cx': ['cs', 'cx', '고객', '상담', '운영', '고객지원'],
  'education': ['교육', '강사', '훈련', '멘토링', '과정'],
  'consulting': ['컨설팅', '컨설턴트', 'consulting']
}

INDUSTRY_KEYWORDS = {
  'it_software': ['소프트웨어', 'it', 'saas', '플랫폼', '앱', '웹', '클라우드'],
  'ai_data': ['ai', '데이터', '머신러닝', '딥러닝', '인공지능'],
  'finance_fintech': ['금융', '핀테크', '증권', '자산운용', '보험', '은행'],
  'retail_ecommerce': ['유통', '이커머스', '커머스', '리테일', '쇼핑몰'],
  'bio_healthcare': ['바이오', '헬스케어', '제약', '의료기기', '임상'],
  'semiconductor': ['반도체', '메모리', '파운드리', 'wafer', '공정기술'],
  'automotive': ['자동차', '모빌리티', '차량', '전장'],
  'beauty_cosmetics': ['뷰티', '화장품', '코스메틱'],
  'electronics': ['전자', '전기', '가전', '디스플레이'],
  'aerospace_defense': ['항공', '방산', '우주', '국방'],
  'logistics': ['물류', '운송', '항만', '무역'],
  'battery_energy': ['배터리', '에너지', '전지', 'ess'],
  'machinery_heavy': ['기계', '중공업', '장비', '플랜트'],
  'chemical_materials': ['화학', '소재', '재료', '고분자'],
  'construction': ['건설', '토목', '건축'],
  'fashion': ['패션', '의류', '섬유'],
  'medical': ['의료', '병원', '간호', '보건'],
  'robotics': ['로봇', '로보틱스', '자동화']
}

SKILL_KEYWORDS = [
  'Python', 'Java', 'JavaScript', 'TypeScript', 'React', 'Spring', 'SQL', 'AWS', 'Docker', 'Kubernetes',
  'Excel', 'VBA', 'Tableau', 'Power BI', 'Pandas', 'PyTorch', 'TensorFlow', '머신러닝', '딥러닝',
  '데이터 분석', '통계', 'A/B 테스트', 'Figma', 'Photoshop', 'Illustrator', 'CAD', 'CAM', 'PLC',
  'ERP', 'SCM', '품질관리', '공정개선', '실험계획법', 'DOE', '회계', '재무', '세무', '마케팅', '영업'
]

COMPETENCY_KEYWORDS = [
  '문제해결', '분석', '협업', '커뮤니케이션', '리더십', '실행력', '기획', '전략', '성장',
  '학습', '자동화', '개선', '성과', '조율', '책임', '고객지향', '데이터 기반'
]


def infer_taxonomy(text: str, mapping: Dict[str, List[str]], default: str = 'unknown') -> str:
  lowered = normalize_text(text).lower()
  scores = {}
  for label, keywords in mapping.items():
    scores[label] = sum(1 for keyword in keywords if keyword.lower() in lowered)
  best_label, best_score = max(scores.items(), key=lambda item: item[1])
  return best_label if best_score > 0 else default


def extract_keywords(text: str, candidates: List[str], max_items: int = 12) -> List[str]:
  lowered = normalize_text(text).lower()
  found = []
  for candidate in candidates:
    if candidate.lower() in lowered:
      found.append(candidate)
  return unique_keep_order(found, max_items=max_items)


def mojibake_ratio(text: str) -> float:
  text = normalize_text(text)
  if not text:
    return 0.0
  suspicious = sum(1 for ch in text if ch == '�' or (ord(ch) < 32 and ch not in '\n\t\r'))
  ascii_noise = len(re.findall(r'[A-Za-z]{1,2} [A-Za-z]{1,2} [A-Za-z]{1,2}', text))
  return min(1.0, (suspicious + ascii_noise) / max(1, len(text)))


## 2. 데이터 로드 및 User/JD 입력 구성

기존 baseline은 유저의 여러 경험 행을 모두 이어 붙였습니다. 이 노트북에서는 같은 자기소개서가 여러 경험 행에 반복되는 문제를 줄이기 위해 `resume_writingId`와 `draftContent` 기준으로 중복을 제거합니다.

In [ ]:
df_user = pd.read_csv(USER_DATA_PATH, dtype=str, low_memory=False).fillna('')
df_jd = pd.read_csv(JD_DATA_PATH, dtype=str, low_memory=False).fillna('')

print('user_data:', df_user.shape, 'unique users:', df_user['userId'].nunique())
print('jd_data:', df_jd.shape, 'unique jobs:', df_jd['job_id'].nunique())


In [ ]:
USER_BASE_COLUMNS = [
  'userId', 'education', 'university', 'major',
  'interestedIndustries_1', 'interestedIndustries_2', 'interestedIndustries_3',
  'interestedJobs_1', 'interestedJobs_2', 'interestedJobs_3'
]

USER_TEXT_COLUMNS = [
  'resume_writingId', 'question', 'draftContent', 'categoryName', 'Title',
  'Situation', 'Task', 'Action', 'Reason', 'Result',
  'ability_0_keyword', 'ability_0_name', 'ability_0_definition', 'ability_0_reason',
  'ability_1_keyword', 'ability_1_name', 'ability_1_definition', 'ability_1_reason',
  'ability_2_keyword', 'ability_2_name', 'ability_2_definition', 'ability_2_reason'
]


def build_user_record(group: pd.DataFrame) -> Dict[str, Any]:
  base = {column: normalize_text(group[column].iloc[0]) for column in USER_BASE_COLUMNS if column in group.columns}
  dedupe_subset = [column for column in ['resume_writingId', 'draftContent'] if column in group.columns]
  if dedupe_subset:
    unique_rows = group.drop_duplicates(subset=dedupe_subset, keep='first')
  else:
    unique_rows = group.drop_duplicates(keep='first')

  questions = unique_join(unique_rows.get('question', []), max_items=8)
  drafts = unique_join(unique_rows.get('draftContent', []), sep='\n', max_items=8)
  star_parts = []
  for _, row in unique_rows.head(12).iterrows():
    star = ' '.join(normalize_text(row.get(col, '')) for col in ['Title', 'Situation', 'Task', 'Action', 'Reason', 'Result'])
    if star.strip():
      star_parts.append(star)
  star_text = unique_join(star_parts, sep='\n', max_items=12)

  ability_values = []
  for idx in range(3):
    for suffix in ['keyword', 'name', 'definition', 'reason']:
      col = f'ability_{idx}_{suffix}'
      if col in unique_rows.columns:
        ability_values.extend(unique_rows[col].tolist())
  ability_text = unique_join(ability_values, max_items=40)

  profile_input = '\n'.join([
    f"학력/전공: {base.get('education', '')} {base.get('university', '')} {base.get('major', '')}",
    f"희망 산업: {base.get('interestedIndustries_1', '')}, {base.get('interestedIndustries_2', '')}, {base.get('interestedIndustries_3', '')}",
    f"희망 직무: {base.get('interestedJobs_1', '')}, {base.get('interestedJobs_2', '')}, {base.get('interestedJobs_3', '')}",
    f"자기소개서 문항: {questions}",
    f"자기소개서 본문: {drafts}",
    f"STAR 경험 근거: {star_text}",
    f"기존 추출 역량: {ability_text}",
  ])

  baseline_text = ' '.join([
    base.get('major', ''), base.get('university', ''), base.get('education', ''),
    base.get('interestedIndustries_1', ''), base.get('interestedJobs_1', ''),
    ability_text, drafts
  ])

  return {
    **base,
    'deduped_draft_count': int(unique_rows['draftContent'].astype(bool).sum()) if 'draftContent' in unique_rows.columns else len(unique_rows),
    'raw_profile_text': normalize_text(profile_input, max_chars=12000),
    'baseline_text': normalize_text(baseline_text, max_chars=12000),
  }


user_records = [build_user_record(group) for _, group in df_user.groupby('userId', sort=True)]
df_user_agg = pd.DataFrame(user_records)

print('aggregated users:', df_user_agg.shape)
df_user_agg.head(3)


In [ ]:
JD_COLUMNS = ['job_id', 'company_name', 'title', 'job_types', 'duties_clean', 'skills_clean', 'benefits_clean', 'detail_text_clean']


def build_jd_raw_text(row: pd.Series) -> str:
  return '\n'.join([
    f"회사명: {normalize_text(row.get('company_name', ''))}",
    f"공고명: {normalize_text(row.get('title', ''))}",
    f"고용/경력 태그: {normalize_text(row.get('job_types', ''))}",
    f"주요 업무: {normalize_text(row.get('duties_clean', ''), max_chars=2500)}",
    f"자격/스킬: {normalize_text(row.get('skills_clean', ''), max_chars=2500)}",
    f"상세 공고: {normalize_text(row.get('detail_text_clean', ''), max_chars=5500)}",
  ])


def prepare_jd_sample(df: pd.DataFrame, sample_size: int = JD_SAMPLE_SIZE) -> pd.DataFrame:
  work = df.copy()
  work['jd_raw_text'] = work.apply(build_jd_raw_text, axis=1)
  work['jd_text_len'] = work['jd_raw_text'].str.len()
  work['detail_len'] = work['detail_text_clean'].astype(str).str.len()
  work['title_len'] = work['title'].astype(str).str.len()

  # 너무 짧은 공고는 의미 매칭 후보로 부적합하므로 우선 제외하되, 부족하면 전체에서 보충합니다.
  eligible = work[(work['jd_text_len'] >= 120) & (work['title_len'] > 0)].copy()
  if len(eligible) >= sample_size:
    sampled = eligible.sample(sample_size, random_state=RANDOM_SEED)
  else:
    sampled = pd.concat([eligible, work.drop(index=eligible.index).sample(sample_size - len(eligible), random_state=RANDOM_SEED)])

  sampled = sampled.sort_values('job_id').reset_index(drop=True)
  sampled['baseline_text'] = sampled['jd_raw_text'].map(lambda text: normalize_text(text, max_chars=12000))
  return sampled


df_jd_sample = prepare_jd_sample(df_jd, JD_SAMPLE_SIZE)
print('sampled JD:', df_jd_sample.shape)
df_jd_sample[['job_id', 'company_name', 'title', 'jd_text_len']].head(5)


## 3. Gemini structured output 스키마

User와 JD를 같은 의미 공간에서 비교하려면 원문을 그대로 임베딩하지 않고, 직무 관련 정보만 추출한 요약 profile을 만들어야 합니다. 아래 JSON schema는 Gemini가 항상 같은 키를 반환하도록 강제하기 위한 구조입니다.

In [ ]:
USER_PROFILE_SCHEMA = {
  'type': 'object',
  'properties': {
    'target_jobs': {'type': 'array', 'items': {'type': 'string'}, 'maxItems': 5},
    'target_industries': {'type': 'array', 'items': {'type': 'string'}, 'maxItems': 5},
    'hard_skills': {'type': 'array', 'items': {'type': 'string'}, 'maxItems': 12},
    'tools': {'type': 'array', 'items': {'type': 'string'}, 'maxItems': 12},
    'achievement_evidence': {'type': 'array', 'items': {'type': 'string'}, 'maxItems': 8},
    'competencies': {'type': 'array', 'items': {'type': 'string'}, 'maxItems': 10},
    'job_relevance_summary': {'type': 'string'},
    'embedding_summary': {'type': 'string'},
    'quality_flags': {'type': 'array', 'items': {'type': 'string'}, 'maxItems': 8}
  },
  'required': [
    'target_jobs', 'target_industries', 'hard_skills', 'tools', 'achievement_evidence',
    'competencies', 'job_relevance_summary', 'embedding_summary', 'quality_flags'
  ],
  'additionalProperties': False
}

JD_PROFILE_SCHEMA = {
  'type': 'object',
  'properties': {
    'posting_type': {'type': 'string', 'enum': ['job_posting', 'training_program', 'internship_program', 'company_promotion', 'low_quality', 'unknown']},
    'job_role': {'type': 'string', 'enum': JOB_TAXONOMY},
    'industry': {'type': 'string', 'enum': INDUSTRY_TAXONOMY},
    'experience_level': {'type': 'string', 'enum': ['new', 'experienced', 'both', 'intern', 'unknown']},
    'core_duties': {'type': 'array', 'items': {'type': 'string'}, 'maxItems': 8},
    'required_skills': {'type': 'array', 'items': {'type': 'string'}, 'maxItems': 12},
    'preferred_skills': {'type': 'array', 'items': {'type': 'string'}, 'maxItems': 12},
    'soft_competencies': {'type': 'array', 'items': {'type': 'string'}, 'maxItems': 10},
    'culture_fit': {'type': 'array', 'items': {'type': 'string'}, 'maxItems': 8},
    'jd_summary': {'type': 'string'},
    'embedding_summary': {'type': 'string'},
    'quality_flags': {'type': 'array', 'items': {'type': 'string'}, 'maxItems': 8}
  },
  'required': [
    'posting_type', 'job_role', 'industry', 'experience_level', 'core_duties', 'required_skills',
    'preferred_skills', 'soft_competencies', 'culture_fit', 'jd_summary', 'embedding_summary', 'quality_flags'
  ],
  'additionalProperties': False
}

JUDGE_SCHEMA = {
  'type': 'object',
  'properties': {
    'relevance': {'type': 'integer', 'minimum': 0, 'maximum': 4},
    'rationale': {'type': 'string'},
    'matched_evidence': {'type': 'array', 'items': {'type': 'string'}, 'maxItems': 5},
    'missing_requirements': {'type': 'array', 'items': {'type': 'string'}, 'maxItems': 5},
    'confidence': {'type': 'number', 'minimum': 0, 'maximum': 1}
  },
  'required': ['relevance', 'rationale', 'matched_evidence', 'missing_requirements', 'confidence'],
  'additionalProperties': False
}


In [ ]:
def user_profile_prompt(raw_text: str) -> str:
  return f'''
너는 한국어 채용 매칭용 전처리 모델이다. 아래 지원자 자기소개서/경험 데이터에서 직무 관련 정보만 추출하라.

규칙:
- 인성 표현, 감성적 수사, 반복 문장은 제거한다.
- 실제 업무 역량, 기술, 도구, 정량 성과, STAR 근거만 남긴다.
- 원문에 없는 능력을 추측하지 않는다.
- 출력은 schema에 맞는 JSON만 반환한다.

지원자 원문:
{raw_text}
'''.strip()


def jd_profile_prompt(raw_text: str) -> str:
  return f'''
너는 한국어 JD 매칭용 전처리 모델이다. 아래 채용공고에서 실제 직무 관련 정보만 추출하라.

규칙:
- 회사 홍보, 복리후생 나열, 지원 안내, 접수 기간, 일반 교육 홍보는 직무 요건에서 제외한다.
- 교육과정/부트캠프/멘토링/동아리 모집이면 posting_type으로 표시한다.
- 깨진 인코딩, 과도한 템플릿, 정보 부족은 quality_flags에 표시한다.
- job_role과 industry는 제공된 taxonomy 중 하나만 고른다.
- 출력은 schema에 맞는 JSON만 반환한다.

JD 원문:
{raw_text}
'''.strip()


def judge_prompt(user_profile: Dict[str, Any], jd_profile: Dict[str, Any]) -> str:
  user_payload = json.dumps(user_profile, ensure_ascii=False, indent=2)
  jd_payload = json.dumps(jd_profile, ensure_ascii=False, indent=2)
  return f'''
너는 한국어 채용 매칭 평가자다. 추천 모델의 점수나 순위는 보지 말고, 아래 지원자 profile과 JD profile만 보고 적합도를 0~4점으로 평가하라.

평가기준:
4 = 직무/산업/핵심스킬/성과근거가 매우 잘 맞음
3 = 주요 직무와 역량이 대체로 맞음
2 = 일부 관련은 있으나 핵심 요건이 부족함
1 = 약한 관련만 있음
0 = 직무적으로 부적합하거나 공고 품질이 낮음

지원자 profile:
{user_payload}

JD profile:
{jd_payload}

JSON만 반환하라.
'''.strip()


## 4. Gemini 호출 및 fallback profile

실제 Gemini 호출이 가능한 환경이면 structured output을 사용합니다. 지금처럼 API 키가 없거나 `google-genai`가 설치되지 않은 환경에서도 노트북 전체 구조와 지표 계산을 검증할 수 있도록 heuristic fallback을 제공합니다.

In [ ]:
def get_genai_client():
  if not USE_GEMINI:
    return None
  return genai.Client(api_key=GEMINI_API_KEY)


def call_gemini_json(prompt: str, schema: Dict[str, Any], model_name: str = GEMINI_MODEL, max_retries: int = 3) -> Dict[str, Any]:
  if not USE_GEMINI:
    raise RuntimeError('Gemini API를 사용할 수 없습니다. google-genai 설치와 API 키를 확인하세요.')

  client = get_genai_client()
  last_error = None
  for attempt in range(max_retries):
    try:
      response = client.models.generate_content(
        model=model_name,
        contents=prompt,
        config=types.GenerateContentConfig(
          response_mime_type='application/json',
          response_json_schema=schema,
          temperature=0.0,
        ),
      )
      return json.loads(response.text)
    except Exception as exc:
      last_error = exc
      time.sleep(1.5 * (attempt + 1))
  raise RuntimeError(f'Gemini 호출 실패: {last_error}')


def heuristic_user_profile(row: pd.Series) -> Dict[str, Any]:
  raw_text = normalize_text(row.get('raw_profile_text', ''), max_chars=12000)
  target_jobs = unique_keep_order([row.get('interestedJobs_1', ''), row.get('interestedJobs_2', ''), row.get('interestedJobs_3', '')], max_items=5)
  target_industries = unique_keep_order([row.get('interestedIndustries_1', ''), row.get('interestedIndustries_2', ''), row.get('interestedIndustries_3', '')], max_items=5)
  hard_skills = extract_keywords(raw_text, SKILL_KEYWORDS, max_items=12)
  tools = extract_keywords(raw_text, ['Python', 'Java', 'React', 'Spring', 'SQL', 'AWS', 'Docker', 'Excel', 'VBA', 'Figma', 'CAD', 'ERP'], max_items=12)
  competencies = extract_keywords(raw_text, COMPETENCY_KEYWORDS, max_items=10)

  evidence_candidates = []
  for pattern in [r'[^.。!?]*\d+[%배시간분년개월][^.。!?]*', r'[^.。!?]*(개선|달성|구축|자동화|분석|최적화)[^.。!?]*']:
    evidence_candidates.extend(re.findall(pattern, raw_text))
  achievement_evidence = unique_keep_order(evidence_candidates, max_items=8)

  summary_parts = [
    f"희망직무: {', '.join(target_jobs)}",
    f"희망산업: {', '.join(target_industries)}",
    f"전공: {normalize_text(row.get('major', ''))}",
    f"하드스킬: {', '.join(hard_skills)}",
    f"도구: {', '.join(tools)}",
    f"역량: {', '.join(competencies)}",
    f"성과근거: {'; '.join(achievement_evidence[:4])}",
  ]

  return {
    'target_jobs': target_jobs,
    'target_industries': target_industries,
    'hard_skills': hard_skills,
    'tools': tools,
    'achievement_evidence': achievement_evidence,
    'competencies': competencies,
    'job_relevance_summary': normalize_text(' '.join(summary_parts), max_chars=1500),
    'embedding_summary': normalize_text(' '.join(summary_parts), max_chars=1500),
    'quality_flags': ['heuristic_fallback'],
  }


def heuristic_jd_profile(row: pd.Series) -> Dict[str, Any]:
  raw_text = normalize_text(row.get('jd_raw_text', ''), max_chars=12000)
  title = normalize_text(row.get('title', ''))
  combined = f'{title} {raw_text}'
  job_role = infer_taxonomy(combined, JOB_KEYWORDS)
  industry = infer_taxonomy(combined, INDUSTRY_KEYWORDS)
  lowered = combined.lower()

  if any(keyword in combined for keyword in ['교육과정', '훈련', '수강', '멘토링', '학원', '교육기관']):
    posting_type = 'training_program'
  elif any(keyword in combined for keyword in ['동아리', '서포터즈', '프로젝트 모집']):
    posting_type = 'internship_program'
  elif len(raw_text) < 150 or mojibake_ratio(raw_text) > 0.05:
    posting_type = 'low_quality'
  else:
    posting_type = 'job_posting'

  if 'intern' in lowered or '인턴' in combined:
    experience_level = 'intern'
  elif 'experienced' in lowered and 'new' in lowered:
    experience_level = 'both'
  elif '경력' in combined or 'experienced' in lowered:
    experience_level = 'experienced'
  elif '신입' in combined or 'new' in lowered:
    experience_level = 'new'
  else:
    experience_level = 'unknown'

  required_skills = extract_keywords(combined, SKILL_KEYWORDS, max_items=12)
  soft_competencies = extract_keywords(combined, COMPETENCY_KEYWORDS, max_items=10)
  sentence_candidates = re.split(r'[.|。|!|?]|\s{2,}', raw_text)
  core_duties = unique_keep_order([s for s in sentence_candidates if any(k in s for k in ['업무', '개발', '운영', '관리', '분석', '기획', '품질', '영업', '마케팅', '생산'])], max_items=8)

  quality_flags = ['heuristic_fallback']
  if posting_type != 'job_posting':
    quality_flags.append(posting_type)
  if mojibake_ratio(raw_text) > 0.05:
    quality_flags.append('possible_mojibake')
  if len(required_skills) == 0 and len(core_duties) == 0:
    quality_flags.append('low_job_signal')

  summary_parts = [
    f"직무: {job_role}",
    f"산업: {industry}",
    f"경력수준: {experience_level}",
    f"핵심업무: {'; '.join(core_duties[:4])}",
    f"필수스킬: {', '.join(required_skills)}",
    f"소프트역량: {', '.join(soft_competencies)}",
  ]

  return {
    'posting_type': posting_type,
    'job_role': job_role,
    'industry': industry,
    'experience_level': experience_level,
    'core_duties': core_duties,
    'required_skills': required_skills,
    'preferred_skills': [],
    'soft_competencies': soft_competencies,
    'culture_fit': [],
    'jd_summary': normalize_text(' '.join(summary_parts), max_chars=1500),
    'embedding_summary': normalize_text(' '.join(summary_parts), max_chars=1500),
    'quality_flags': quality_flags,
  }


In [ ]:
def profile_cache_path(kind: str, record_id: str, raw_text: str) -> Path:
  digest = stable_hash(kind, record_id, raw_text, PROFILE_ENGINE)
  return CACHE_DIR / kind / f'{record_id}_{digest}.json'


def build_user_profile(row: pd.Series) -> Dict[str, Any]:
  user_id = normalize_text(row.get('userId', 'unknown'))
  raw_text = normalize_text(row.get('raw_profile_text', ''), max_chars=12000)
  cache_path = profile_cache_path('user_profile', user_id, raw_text)
  expected_model = GEMINI_MODEL if USE_GEMINI else 'heuristic_fallback'
  cached = load_json(cache_path)
  if cached is not None and cached.get('_model') == expected_model:
    return cached

  if USE_GEMINI:
    profile = call_gemini_json(user_profile_prompt(raw_text), USER_PROFILE_SCHEMA)
  else:
    profile = heuristic_user_profile(row)

  profile['_record_id'] = user_id
  profile['_model'] = GEMINI_MODEL if USE_GEMINI else 'heuristic_fallback'
  save_json(cache_path, profile)
  return profile


def build_jd_profile(row: pd.Series) -> Dict[str, Any]:
  job_id = normalize_text(row.get('job_id', 'unknown'))
  raw_text = normalize_text(row.get('jd_raw_text', ''), max_chars=12000)
  cache_path = profile_cache_path('jd_profile', job_id, raw_text)
  expected_model = GEMINI_MODEL if USE_GEMINI else 'heuristic_fallback'
  cached = load_json(cache_path)
  if cached is not None and cached.get('_model') == expected_model:
    return cached

  if USE_GEMINI:
    profile = call_gemini_json(jd_profile_prompt(raw_text), JD_PROFILE_SCHEMA)
  else:
    profile = heuristic_jd_profile(row)

  profile['_record_id'] = job_id
  profile['_model'] = GEMINI_MODEL if USE_GEMINI else 'heuristic_fallback'
  save_json(cache_path, profile)
  return profile

def build_profiles(df: pd.DataFrame, builder, label: str, max_workers: int = GEMINI_MAX_WORKERS) -> List[Dict[str, Any]]:
  total = len(df)
  items = list(df.iterrows())
  profiles = [None] * total
  workers = max(1, int(max_workers))

  if workers == 1:
    for pos, (_, row) in enumerate(items):
      if pos % 100 == 0:
        print(f'{label} {pos}/{total}')
      profiles[pos] = builder(row)
    return profiles

  print(f'{label}: parallel workers={workers}')
  with ThreadPoolExecutor(max_workers=workers) as executor:
    futures = {executor.submit(builder, row): pos for pos, (_, row) in enumerate(items)}
    for done_count, future in enumerate(as_completed(futures), start=1):
      pos = futures[future]
      profiles[pos] = future.result()
      if done_count == 1 or done_count % 100 == 0 or done_count == total:
        print(f'{label} {done_count}/{total}')

  return profiles



## 5. Dry-run: schema와 캐시 검증

처음에는 전체 1,000개 JD를 처리하기 전에 User 5명과 JD 5개만 실행합니다. 캐시가 정상이라면 같은 셀을 다시 실행해도 API 호출 없이 같은 결과가 반환됩니다.

In [ ]:
dry_user_profiles = [build_user_profile(row) for _, row in df_user_agg.head(5).iterrows()]
dry_jd_profiles = [build_jd_profile(row) for _, row in df_jd_sample.head(5).iterrows()]

print('dry user profiles:', len(dry_user_profiles))
print('dry jd profiles:', len(dry_jd_profiles))
print(json.dumps(dry_user_profiles[0], ensure_ascii=False, indent=2)[:1200])
print(json.dumps(dry_jd_profiles[0], ensure_ascii=False, indent=2)[:1200])


## 6. 전체 샘플 Profile 생성

전체 유저와 deterministic JD 샘플에 대해 profile을 생성합니다. 기본 JD 샘플은 1,000개이며, `JD_SAMPLE_SIZE` 환경변수로 조정할 수 있습니다. Gemini API를 사용할 경우 시간이 걸릴 수 있고, 같은 입력은 캐시를 재사용합니다. `GEMINI_MAX_WORKERS`는 기본 1로 두었고 API 한도에 맞춰 조심스럽게 올릴 수 있습니다. 결과는 `data/gemini_profile_outputs`에도 저장합니다.

In [ ]:
if USE_GEMINI:
  print(f'Gemini profile 생성 시작: users={len(df_user_agg)}, max_workers={GEMINI_MAX_WORKERS}')

user_profiles = build_profiles(df_user_agg, build_user_profile, 'user profile')

df_user_profiles = pd.concat(
  [df_user_agg.reset_index(drop=True), pd.json_normalize(user_profiles).add_prefix('profile_')],
  axis=1
)

user_profiles_path = OUTPUT_DIR / 'user_profiles.csv'
df_user_profiles.to_csv(user_profiles_path, index=False, encoding='utf-8-sig')
print('saved:', user_profiles_path)
df_user_profiles.head(2)


In [ ]:
if USE_GEMINI:
  print(f'Gemini JD profile 생성 시작: jds={len(df_jd_sample)}, max_workers={GEMINI_MAX_WORKERS}')

jd_profiles = build_profiles(df_jd_sample, build_jd_profile, 'jd profile')

df_jd_profiles = pd.concat(
  [df_jd_sample.reset_index(drop=True), pd.json_normalize(jd_profiles).add_prefix('profile_')],
  axis=1
)

jd_profiles_path = OUTPUT_DIR / 'jd_profiles_sample1000.csv'
df_jd_profiles.to_csv(jd_profiles_path, index=False, encoding='utf-8-sig')
print('saved:', jd_profiles_path)
df_jd_profiles[['job_id', 'company_name', 'title', 'profile_posting_type', 'profile_job_role', 'profile_industry', 'profile_quality_flags']].head(5)


## 7. Embedding 생성

Baseline A는 기존 방식처럼 원문 텍스트를 임베딩합니다. Model B는 Gemini/heuristic profile의 `embedding_summary`만 임베딩합니다. 임베딩 모델 로드가 실패하면 노트북 검증을 위해 TF-IDF fallback을 사용합니다.

In [ ]:
def normalize_embedding_matrix(matrix: np.ndarray) -> np.ndarray:
  matrix = np.asarray(matrix, dtype='float32')
  norm = np.linalg.norm(matrix, axis=1, keepdims=True)
  norm[norm == 0] = 1.0
  return matrix / norm


def encode_texts(texts: List[str], model_name: str = EMBEDDING_MODEL_NAME, batch_size: int = 32) -> Tuple[np.ndarray, str]:
  texts = [normalize_text(text) for text in texts]
  if HAS_SENTENCE_TRANSFORMERS:
    try:
      model = SentenceTransformer(model_name)
      embeddings = model.encode(texts, show_progress_bar=True, normalize_embeddings=True, batch_size=batch_size)
      return np.asarray(embeddings, dtype='float32'), model_name
    except Exception as exc:
      print('SentenceTransformer 로드/임베딩 실패. TF-IDF fallback 사용:', exc)

  vectorizer = TfidfVectorizer(max_features=4096, ngram_range=(1, 2), min_df=1)
  sparse = vectorizer.fit_transform(texts)
  return normalize_embedding_matrix(sparse.toarray()), 'tfidf_fallback'


def search_topk(query_embeddings: np.ndarray, doc_embeddings: np.ndarray, k: int = 10) -> Tuple[np.ndarray, np.ndarray]:
  query_embeddings = normalize_embedding_matrix(query_embeddings)
  doc_embeddings = normalize_embedding_matrix(doc_embeddings)
  k = min(k, len(doc_embeddings))
  if HAS_FAISS:
    index = faiss.IndexFlatIP(doc_embeddings.shape[1])
    index.add(doc_embeddings.astype('float32'))
    return index.search(query_embeddings.astype('float32'), k)

  sims = query_embeddings @ doc_embeddings.T
  indices = np.argsort(-sims, axis=1)[:, :k]
  distances = np.take_along_axis(sims, indices, axis=1)
  return distances.astype('float32'), indices.astype('int64')


In [ ]:
baseline_user_texts = df_user_profiles['baseline_text'].fillna('').tolist()
baseline_jd_texts = df_jd_profiles['baseline_text'].fillna('').tolist()

profile_user_texts = df_user_profiles['profile_embedding_summary'].fillna('').tolist()
profile_jd_texts = df_jd_profiles['profile_embedding_summary'].fillna('').tolist()

all_baseline_texts = baseline_user_texts + baseline_jd_texts
all_profile_texts = profile_user_texts + profile_jd_texts

baseline_all_embeddings, baseline_encoder_name = encode_texts(all_baseline_texts)
profile_all_embeddings, profile_encoder_name = encode_texts(all_profile_texts)

user_count = len(df_user_profiles)
jd_count = len(df_jd_profiles)

baseline_user_embeddings = baseline_all_embeddings[:user_count]
baseline_jd_embeddings = baseline_all_embeddings[user_count:]
profile_user_embeddings = profile_all_embeddings[:user_count]
profile_jd_embeddings = profile_all_embeddings[user_count:]

print('baseline encoder:', baseline_encoder_name, baseline_user_embeddings.shape, baseline_jd_embeddings.shape)
print('profile encoder:', profile_encoder_name, profile_user_embeddings.shape, profile_jd_embeddings.shape)


## 8. Baseline A와 Model B 검색

두 모델 모두 같은 JD 샘플 1,000개를 대상으로 Faiss cosine search를 수행합니다. `normalize_embeddings=True`이므로 inner product는 cosine similarity와 같습니다.

In [ ]:
TOPK_SEARCH = 100

baseline_distances, baseline_indices = search_topk(baseline_user_embeddings, baseline_jd_embeddings, k=TOPK_SEARCH)
profile_distances, profile_indices = search_topk(profile_user_embeddings, profile_jd_embeddings, k=TOPK_SEARCH)

print('baseline search:', baseline_distances.shape, baseline_indices.shape)
print('profile search:', profile_distances.shape, profile_indices.shape)


In [ ]:
def make_search_results(distances: np.ndarray, indices: np.ndarray, model_name: str) -> pd.DataFrame:
  rows = []
  for user_idx in range(distances.shape[0]):
    user_id = df_user_profiles.iloc[user_idx]['userId']
    for rank, (score, jd_idx) in enumerate(zip(distances[user_idx], indices[user_idx]), start=1):
      jd_row = df_jd_profiles.iloc[int(jd_idx)]
      rows.append({
        'model': model_name,
        'user_idx': user_idx,
        'userId': user_id,
        'rank': rank,
        'job_idx': int(jd_idx),
        'job_id': jd_row['job_id'],
        'company_name': jd_row['company_name'],
        'title': jd_row['title'],
        'score': float(score),
      })
  return pd.DataFrame(rows)

baseline_results = make_search_results(baseline_distances, baseline_indices, 'baseline_raw_sbert')
profile_results = make_search_results(profile_distances, profile_indices, 'gemini_profile_sbert')

search_results_path = OUTPUT_DIR / 'search_results_top100.csv'
pd.concat([baseline_results, profile_results], ignore_index=True).to_csv(search_results_path, index=False, encoding='utf-8-sig')
print('saved:', search_results_path)
baseline_results.head(5)


## 9. Model C: profile-weighted score

Model C는 단일 cosine score만 쓰지 않고, 역할 의미 유사도와 구조화 profile overlap을 함께 사용합니다. 초기 가중치는 계획서 기준입니다.

In [ ]:
WEIGHTS = {
  'role_semantic': 0.35,
  'hard_skill': 0.20,
  'competency': 0.15,
  'achievement': 0.10,
  'industry': 0.10,
  'quality_adjustment': 0.10,
}


def list_from_profile(row: pd.Series, column: str) -> List[str]:
  value = row.get(column, [])
  if isinstance(value, list):
    return value
  if isinstance(value, str):
    try:
      parsed = json.loads(value.replace("'", '"'))
      if isinstance(parsed, list):
        return [normalize_text(v) for v in parsed if normalize_text(v)]
    except Exception:
      pass
  return safe_list(value)


def industry_score(user_row: pd.Series, jd_row: pd.Series) -> float:
  jd_industry = normalize_text(jd_row.get('profile_industry', 'unknown'))
  if not jd_industry or jd_industry == 'unknown':
    return 0.0
  primary = normalize_text(user_row.get('interestedIndustries_1', ''))
  secondary = {normalize_text(user_row.get('interestedIndustries_2', '')), normalize_text(user_row.get('interestedIndustries_3', ''))}
  if jd_industry == primary:
    return 1.0
  if jd_industry in secondary:
    return 0.6
  return 0.0


def quality_score(jd_row: pd.Series) -> float:
  posting_type = normalize_text(jd_row.get('profile_posting_type', 'unknown'))
  flags = list_from_profile(jd_row, 'profile_quality_flags')
  if posting_type == 'job_posting':
    base = 1.0
  elif posting_type in {'internship_program', 'unknown'}:
    base = 0.65
  elif posting_type == 'training_program':
    base = 0.35
  else:
    base = 0.2
  if 'possible_mojibake' in flags:
    base -= 0.25
  if 'low_job_signal' in flags:
    base -= 0.25
  return bounded(base)


def weighted_pair_score(user_idx: int, jd_idx: int, role_semantic: float) -> Dict[str, float]:
  user_row = df_user_profiles.iloc[user_idx]
  jd_row = df_jd_profiles.iloc[jd_idx]

  user_skills = list_from_profile(user_row, 'profile_hard_skills') + list_from_profile(user_row, 'profile_tools')
  jd_skills = list_from_profile(jd_row, 'profile_required_skills') + list_from_profile(jd_row, 'profile_preferred_skills')
  user_competencies = list_from_profile(user_row, 'profile_competencies')
  jd_competencies = list_from_profile(jd_row, 'profile_soft_competencies')
  user_achievements = list_from_profile(user_row, 'profile_achievement_evidence')

  hard_skill = jaccard(user_skills, jd_skills)
  competency = jaccard(user_competencies, jd_competencies)
  achievement = bounded(0.5 * hard_skill + 0.5 * min(1.0, len(user_achievements) / 4.0))
  industry = industry_score(user_row, jd_row)
  quality = quality_score(jd_row)
  role_semantic_norm = bounded((role_semantic + 1.0) / 2.0)

  final_score = (
    WEIGHTS['role_semantic'] * role_semantic_norm +
    WEIGHTS['hard_skill'] * hard_skill +
    WEIGHTS['competency'] * competency +
    WEIGHTS['achievement'] * achievement +
    WEIGHTS['industry'] * industry +
    WEIGHTS['quality_adjustment'] * quality
  )

  return {
    'role_semantic': role_semantic_norm,
    'hard_skill': hard_skill,
    'competency': competency,
    'achievement': achievement,
    'industry': industry,
    'quality_adjustment': quality,
    'weighted_score': bounded(final_score),
  }


In [ ]:
# Model C는 profile search 상위 후보와 baseline search 상위 후보를 union한 후보군을 재랭킹합니다.
weighted_rows = []
for user_idx in range(len(df_user_profiles)):
  candidate_indices = list(dict.fromkeys(list(profile_indices[user_idx, :TOPK_SEARCH]) + list(baseline_indices[user_idx, :50])))
  profile_score_map = {int(jd_idx): float(score) for jd_idx, score in zip(profile_indices[user_idx], profile_distances[user_idx])}
  for jd_idx in candidate_indices:
    role_score = profile_score_map.get(int(jd_idx), float(profile_user_embeddings[user_idx] @ profile_jd_embeddings[int(jd_idx)]))
    parts = weighted_pair_score(user_idx, int(jd_idx), role_score)
    jd_row = df_jd_profiles.iloc[int(jd_idx)]
    weighted_rows.append({
      'model': 'profile_weighted_score',
      'user_idx': user_idx,
      'userId': df_user_profiles.iloc[user_idx]['userId'],
      'job_idx': int(jd_idx),
      'job_id': jd_row['job_id'],
      'company_name': jd_row['company_name'],
      'title': jd_row['title'],
      **parts,
    })

weighted_results = pd.DataFrame(weighted_rows)
weighted_results['rank'] = weighted_results.groupby('userId')['weighted_score'].rank(method='first', ascending=False).astype(int)
weighted_results = weighted_results.sort_values(['userId', 'rank']).reset_index(drop=True)

weighted_results_path = OUTPUT_DIR / 'weighted_results.csv'
weighted_results.to_csv(weighted_results_path, index=False, encoding='utf-8-sig')
print('saved:', weighted_results_path)
weighted_results.head(5)


## 10. 추천 결과 비교 함수

한 유저에 대해 기존 baseline, profile embedding, weighted score 결과를 나란히 확인합니다. 직무 무관 문장이 상위 추천을 왜곡하는지 줄었는지를 정성적으로 볼 수 있습니다.

In [ ]:
def show_recommendations(user_id: str, k: int = 5) -> None:
  user_row = df_user_profiles[df_user_profiles['userId'] == user_id]
  if user_row.empty:
    print(f'userId를 찾을 수 없습니다: {user_id}')
    return
  print('=' * 80)
  print('USER:', user_id)
  print('major:', user_row.iloc[0].get('major', ''), '| target job:', user_row.iloc[0].get('interestedJobs_1', ''), '| target industry:', user_row.iloc[0].get('interestedIndustries_1', ''))
  print('profile:', user_row.iloc[0].get('profile_embedding_summary', '')[:600])

  sections = [
    ('Baseline A: raw text SBERT', baseline_results[(baseline_results['userId'] == user_id) & (baseline_results['rank'] <= k)], 'score'),
    ('Model B: Gemini profile SBERT', profile_results[(profile_results['userId'] == user_id) & (profile_results['rank'] <= k)], 'score'),
    ('Model C: profile weighted score', weighted_results[(weighted_results['userId'] == user_id) & (weighted_results['rank'] <= k)], 'weighted_score'),
  ]
  for title, frame, score_col in sections:
    print('\n' + title)
    for _, row in frame.sort_values('rank').iterrows():
      jd = df_jd_profiles.iloc[int(row['job_idx'])]
      print(f"{int(row['rank'])}. {row['company_name']} | {row['title']} | {score_col}={row[score_col]:.4f} | role={jd.get('profile_job_role', '')} | industry={jd.get('profile_industry', '')} | type={jd.get('profile_posting_type', '')}")

show_recommendations(df_user_profiles.iloc[0]['userId'], k=3)


## 11. 100쌍 벤치마크 후보 생성

사용자가 선택한 기준은 “100쌍만 라벨링”입니다. 다만 NDCG@10은 쌍만 있으면 계산할 수 없고, 쿼리별 후보 리스트가 있어야 합니다. 그래서 10명 유저마다 후보 JD 10개를 구성해 총 100쌍을 만듭니다.

In [ ]:
def select_benchmark_users(df: pd.DataFrame, n_users: int = BENCHMARK_USERS) -> List[int]:
  # 희망직무가 다양하도록 직무별 1명씩 먼저 뽑고, 부족하면 랜덤 보충합니다.
  selected = []
  for _, group in df.groupby('interestedJobs_1', sort=True):
    if len(selected) >= n_users:
      break
    selected.append(int(group.sample(1, random_state=RANDOM_SEED).index[0]))
  if len(selected) < n_users:
    remaining = [idx for idx in df.index if idx not in selected]
    selected.extend(pd.Series(remaining).sample(n_users - len(selected), random_state=RANDOM_SEED).astype(int).tolist())
  return selected[:n_users]


def add_unique_candidate(candidates: List[Dict[str, Any]], seen: set, user_idx: int, jd_idx: int, source: str) -> None:
  key = int(jd_idx)
  if key in seen:
    return
  seen.add(key)
  jd_row = df_jd_profiles.iloc[key]
  baseline_score = float(baseline_user_embeddings[user_idx] @ baseline_jd_embeddings[key])
  profile_score = float(profile_user_embeddings[user_idx] @ profile_jd_embeddings[key])
  weighted_parts = weighted_pair_score(user_idx, key, profile_score)
  candidates.append({
    'user_idx': user_idx,
    'userId': df_user_profiles.iloc[user_idx]['userId'],
    'job_idx': key,
    'job_id': jd_row['job_id'],
    'company_name': jd_row['company_name'],
    'title': jd_row['title'],
    'candidate_source': source,
    'baseline_score': baseline_score,
    'profile_score': profile_score,
    'weighted_score': weighted_parts['weighted_score'],
    'jd_job_role': jd_row.get('profile_job_role', ''),
    'jd_industry': jd_row.get('profile_industry', ''),
    'jd_posting_type': jd_row.get('profile_posting_type', ''),
  })


def build_benchmark_pairs() -> pd.DataFrame:
  rng = np.random.default_rng(RANDOM_SEED)
  rows = []
  selected_users = select_benchmark_users(df_user_profiles, BENCHMARK_USERS)

  for user_idx in selected_users:
    candidates = []
    seen = set()

    for jd_idx in baseline_indices[user_idx, :3]:
      add_unique_candidate(candidates, seen, user_idx, int(jd_idx), 'baseline_top')
    for jd_idx in profile_indices[user_idx, :3]:
      add_unique_candidate(candidates, seen, user_idx, int(jd_idx), 'profile_top')

    weighted_top = weighted_results[weighted_results['user_idx'] == user_idx].sort_values('weighted_score', ascending=False).head(3)
    for jd_idx in weighted_top['job_idx'].tolist():
      add_unique_candidate(candidates, seen, user_idx, int(jd_idx), 'weighted_top')

    user_row = df_user_profiles.iloc[user_idx]
    target_industry = normalize_text(user_row.get('interestedIndustries_1', ''))
    target_job = normalize_text(user_row.get('interestedJobs_1', ''))

    weak_positive = df_jd_profiles[
      ((df_jd_profiles['profile_industry'] == target_industry) | (df_jd_profiles['profile_job_role'] == target_job)) &
      (~df_jd_profiles.index.isin(seen))
    ]
    for jd_idx in weak_positive.sample(min(2, len(weak_positive)), random_state=RANDOM_SEED).index.tolist():
      add_unique_candidate(candidates, seen, user_idx, int(jd_idx), 'taxonomy_weak_positive')

    hard_negative_pool = []
    for jd_idx in profile_indices[user_idx, :50]:
      jd = df_jd_profiles.iloc[int(jd_idx)]
      if jd.get('profile_industry', '') != target_industry and jd.get('profile_job_role', '') != target_job:
        hard_negative_pool.append(int(jd_idx))
    for jd_idx in hard_negative_pool[:2]:
      add_unique_candidate(candidates, seen, user_idx, int(jd_idx), 'semantic_hard_negative')

    while len(candidates) < CANDIDATES_PER_USER:
      jd_idx = int(rng.integers(0, len(df_jd_profiles)))
      add_unique_candidate(candidates, seen, user_idx, jd_idx, 'random_negative')

    rows.extend(candidates[:CANDIDATES_PER_USER])

  benchmark = pd.DataFrame(rows)
  benchmark['candidate_rank_input_order'] = benchmark.groupby('userId').cumcount() + 1
  return benchmark


benchmark_pairs = build_benchmark_pairs()
benchmark_pairs_path = OUTPUT_DIR / 'benchmark_pairs_100.csv'
benchmark_pairs.to_csv(benchmark_pairs_path, index=False, encoding='utf-8-sig')
print('saved:', benchmark_pairs_path)
print('shape:', benchmark_pairs.shape)
print(benchmark_pairs.groupby('userId').size().describe())
benchmark_pairs.head(10)


## 12. Gemini Judge 라벨링

Gemini Judge는 모델 점수와 rank를 보지 않고 User/JD profile만 보고 0~4 적합도 점수를 부여합니다. API를 사용할 수 없으면 같은 벤치마크 파이프라인 검증을 위해 heuristic relevance를 사용합니다.

In [ ]:
def heuristic_judge(user_idx: int, jd_idx: int) -> Dict[str, Any]:
  user_row = df_user_profiles.iloc[user_idx]
  jd_row = df_jd_profiles.iloc[jd_idx]
  user_skills = list_from_profile(user_row, 'profile_hard_skills') + list_from_profile(user_row, 'profile_tools')
  jd_skills = list_from_profile(jd_row, 'profile_required_skills') + list_from_profile(jd_row, 'profile_preferred_skills')
  role_match = 1.0 if normalize_text(user_row.get('interestedJobs_1', '')) == normalize_text(jd_row.get('profile_job_role', '')) else 0.0
  industry_match = industry_score(user_row, jd_row)
  skill_match = jaccard(user_skills, jd_skills)
  quality = quality_score(jd_row)
  raw = 0.35 * role_match + 0.25 * industry_match + 0.25 * skill_match + 0.15 * quality
  if quality < 0.35:
    raw *= 0.6
  relevance = int(np.clip(round(raw * 4), 0, 4))
  return {
    'relevance': relevance,
    'rationale': 'heuristic fallback: role/industry/skill/quality overlap 기반 점수',
    'matched_evidence': user_skills[:3],
    'missing_requirements': [skill for skill in jd_skills if skill not in user_skills][:3],
    'confidence': 0.45,
  }


def judge_cache_path(user_id: str, job_id: str) -> Path:
  digest = stable_hash('judge', user_id, job_id, PROFILE_ENGINE)
  return CACHE_DIR / 'judge' / f'{user_id}_{job_id}_{digest}.json'


def judge_pair(row: pd.Series) -> Dict[str, Any]:
  user_idx = int(row['user_idx'])
  jd_idx = int(row['job_idx'])
  user_id = normalize_text(row['userId'])
  job_id = normalize_text(row['job_id'])
  cache_path = judge_cache_path(user_id, job_id)
  expected_model = GEMINI_MODEL if USE_GEMINI else 'heuristic_fallback'
  cached = load_json(cache_path)
  if cached is not None and cached.get('_model') == expected_model:
    return cached

  user_profile = {col.replace('profile_', ''): df_user_profiles.iloc[user_idx][col] for col in df_user_profiles.columns if col.startswith('profile_')}
  jd_profile = {col.replace('profile_', ''): df_jd_profiles.iloc[jd_idx][col] for col in df_jd_profiles.columns if col.startswith('profile_')}

  if USE_GEMINI:
    label = call_gemini_json(judge_prompt(user_profile, jd_profile), JUDGE_SCHEMA)
  else:
    label = heuristic_judge(user_idx, jd_idx)

  label['_model'] = GEMINI_MODEL if USE_GEMINI else 'heuristic_fallback'
  save_json(cache_path, label)
  return label


judge_labels = []
for idx, row in benchmark_pairs.iterrows():
  if idx % 20 == 0:
    print(f'judge {idx}/{len(benchmark_pairs)}')
  judge_labels.append(judge_pair(row))

df_judge = pd.json_normalize(judge_labels).add_prefix('judge_')
benchmark_labeled = pd.concat([benchmark_pairs.reset_index(drop=True), df_judge], axis=1)
benchmark_labeled['judge_relevance'] = benchmark_labeled['judge_relevance'].astype(int).clip(0, 4)

benchmark_labeled_path = OUTPUT_DIR / 'benchmark_labeled_100.csv'
benchmark_labeled.to_csv(benchmark_labeled_path, index=False, encoding='utf-8-sig')
print('saved:', benchmark_labeled_path)
benchmark_labeled[['userId', 'job_id', 'candidate_source', 'baseline_score', 'profile_score', 'weighted_score', 'judge_relevance', 'judge_rationale']].head(10)


## 13. Ranking Metrics 계산

NDCG@K는 graded relevance를 순위 할인까지 반영하므로 이번 추천 품질 평가의 핵심 지표입니다. Precision@K는 `relevance >= 3`을 적합으로 보고 상위 K개 중 적합 비율을 계산합니다. MRR@K는 첫 번째 적합 JD가 얼마나 앞에 나오는지를 봅니다.

In [ ]:
def dcg_at_k(relevance: List[float], k: int) -> float:
  rel = np.asarray(relevance[:k], dtype=float)
  if rel.size == 0:
    return 0.0
  gains = np.power(2.0, rel) - 1.0
  discounts = np.log2(np.arange(2, rel.size + 2))
  return float(np.sum(gains / discounts))


def ndcg_at_k(relevance: List[float], k: int) -> float:
  actual = dcg_at_k(relevance, k)
  ideal = dcg_at_k(sorted(relevance, reverse=True), k)
  if ideal == 0:
    return 0.0
  return actual / ideal


def precision_at_k(relevance: List[float], k: int, threshold: int = 3) -> float:
  rel = relevance[:k]
  if not rel:
    return 0.0
  return float(sum(score >= threshold for score in rel) / min(k, len(rel)))


def mrr_at_k(relevance: List[float], k: int, threshold: int = 3) -> float:
  for idx, score in enumerate(relevance[:k], start=1):
    if score >= threshold:
      return 1.0 / idx
  return 0.0


def evaluate_model(frame: pd.DataFrame, score_col: str, model_name: str) -> Dict[str, Any]:
  query_metrics = []
  for user_id, group in frame.groupby('userId'):
    ranked = group.sort_values(score_col, ascending=False)
    relevance = ranked['judge_relevance'].astype(float).tolist()
    query_metrics.append({
      'userId': user_id,
      'ndcg@5': ndcg_at_k(relevance, 5),
      'ndcg@10': ndcg_at_k(relevance, 10),
      'precision@5': precision_at_k(relevance, 5),
      'precision@10': precision_at_k(relevance, 10),
      'mrr@10': mrr_at_k(relevance, 10),
    })

  query_df = pd.DataFrame(query_metrics)
  score_rank = frame[score_col].rank(method='average')
  rel_rank = frame['judge_relevance'].rank(method='average')
  spearman = float(score_rank.corr(rel_rank)) if score_rank.nunique() > 1 and rel_rank.nunique() > 1 else 0.0

  return {
    'model': model_name,
    'score_col': score_col,
    'ndcg@5': query_df['ndcg@5'].mean(),
    'ndcg@10': query_df['ndcg@10'].mean(),
    'precision@5': query_df['precision@5'].mean(),
    'precision@10': query_df['precision@10'].mean(),
    'mrr@10': query_df['mrr@10'].mean(),
    'spearman': spearman,
  }


metrics_rows = [
  evaluate_model(benchmark_labeled, 'baseline_score', 'Baseline A: raw text SBERT'),
  evaluate_model(benchmark_labeled, 'profile_score', 'Model B: Gemini profile SBERT'),
  evaluate_model(benchmark_labeled, 'weighted_score', 'Model C: profile weighted score'),
]
metrics_df = pd.DataFrame(metrics_rows)
metrics_path = OUTPUT_DIR / 'benchmark_metrics.csv'
metrics_df.to_csv(metrics_path, index=False, encoding='utf-8-sig')
print('saved:', metrics_path)
metrics_df


## 14. 검증 체크

계획서의 Test Plan에 맞춰 데이터 수, 캐시, 지표 범위를 자동 확인합니다. 실패하면 assert가 발생하므로 해당 셀을 통과하는 것이 최소 품질 기준입니다.

In [ ]:
assert len(benchmark_labeled) == BENCHMARK_USERS * CANDIDATES_PER_USER, 'benchmark pair 수가 100이 아닙니다.'
assert benchmark_labeled['userId'].nunique() == BENCHMARK_USERS, 'benchmark user 수가 10명이 아닙니다.'
assert benchmark_labeled.groupby('userId').size().eq(CANDIDATES_PER_USER).all(), 'user당 후보 수가 10개가 아닙니다.'

metric_cols_0_1 = ['ndcg@5', 'ndcg@10', 'precision@5', 'precision@10', 'mrr@10']
assert metrics_df[metric_cols_0_1].notna().all().all(), '지표에 NaN이 있습니다.'
assert ((metrics_df[metric_cols_0_1] >= 0) & (metrics_df[metric_cols_0_1] <= 1)).all().all(), '0~1 범위를 벗어난 지표가 있습니다.'
assert metrics_df['spearman'].notna().all(), 'Spearman에 NaN이 있습니다.'
assert ((metrics_df['spearman'] >= -1) & (metrics_df['spearman'] <= 1)).all(), 'Spearman 범위가 잘못되었습니다.'

user_cache_files = list((CACHE_DIR / 'user_profile').glob('*.json'))
jd_cache_files = list((CACHE_DIR / 'jd_profile').glob('*.json'))
judge_cache_files = list((CACHE_DIR / 'judge').glob('*.json'))
print('cache files:', {'user_profile': len(user_cache_files), 'jd_profile': len(jd_cache_files), 'judge': len(judge_cache_files)})
print('검증 통과')


## 15. 정성 비교 샘플

아래 셀은 벤치마크에 포함된 첫 번째 유저의 상위 추천을 비교합니다. 실제 보고서에는 이 결과를 캡처하거나, `benchmark_labeled_100.csv`에서 relevance가 크게 달라진 사례를 골라 분석하면 됩니다.

In [ ]:
example_user_id = benchmark_labeled.iloc[0]['userId']
show_recommendations(example_user_id, k=3)

print('\nBenchmark labels for example user')
benchmark_labeled[benchmark_labeled['userId'] == example_user_id][[
  'candidate_source', 'company_name', 'title', 'baseline_score', 'profile_score', 'weighted_score', 'judge_relevance', 'judge_rationale'
]].sort_values('weighted_score', ascending=False)


## 16. 다음 실험 확장 방향

- `GEMINI_API_KEY`를 설정한 뒤 이 노트북을 다시 실행하면 heuristic fallback 대신 Gemini structured output이 캐시에 저장됩니다.
- 100쌍 benchmark는 v1의 최소 평가셋입니다. 발표/논문용으로는 20명 × 10후보 = 200쌍 이상으로 확장하는 편이 더 안정적입니다.
- `WEIGHTS`는 초기값입니다. benchmark가 누적되면 grid search나 ablation으로 `role_semantic`, `hard_skill`, `industry` 가중치를 조정할 수 있습니다.
- 현재는 JD 샘플 1,000개 기준입니다. 최종 추천 시스템에서는 전체 JD 235,850건을 embedding index로 확장하되, Gemini profile 생성 비용 때문에 우선 품질 좋은 공고만 필터링하는 단계가 필요합니다.